In [ ]:
# ============================================================
# Smart MCQ Solver Challenge - Dummy Baseline Submission
# ============================================================

import pandas as pd
import os

# -----------------------------
# Show all available files
# -----------------------------
print("Available files:\n")

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# -----------------------------
# Update this path if necessary
# -----------------------------
DATA_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge"

# -----------------------------
# Load datasets
# -----------------------------
train = pd.read_csv(f"{DATA_PATH}/train.csv")
test = pd.read_csv(f"{DATA_PATH}/test.csv")
sample_submission = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")

# -----------------------------
# Basic Information
# -----------------------------
print("\nTrain Shape:", train.shape)
print("Test Shape :", test.shape)

print("\nTrain Columns:")
print(train.columns.tolist())

print("\nFirst 5 Training Rows:")
display(train.head())

print("\nFirst 5 Test Rows:")
display(test.head())

print("\nSample Submission:")
display(sample_submission.head())

# -----------------------------
# Dummy Prediction
# Predict A B C for every row
# -----------------------------
submission = sample_submission.copy()
submission["Prediction"] = "A B C"

# -----------------------------
# Save submission
# -----------------------------
submission.to_csv("submission.csv", index=False)

print("\n✅ submission.csv created successfully!")
print("\nFirst 5 Predictions:")
display(submission.head())

# Milestone 2 

In [1]:
!pip install -q transformers
!pip install -q datasets
!pip install -q sentence-transformers

In [2]:
import torch
import numpy as np

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline
)

from sentence_transformers import (
    SentenceTransformer,
    util
)

In [11]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

train = dataset["train"]

# Create new column
def combine(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

train = train.map(combine)

# Find row 51
text = train[51]["combined_text"]

print(text)

# Character length
print(len(text))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
614


In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
print(tokenizer.vocab_size)

30522


In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [15]:
print(tokenizer.sep_token)
print(tokenizer.sep_token_id)

[SEP]
102


In [19]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)
train = dataset["train"]

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Ensure every prompt is a string
prompts = [str(x) if x is not None else "" for x in train["prompt"]]

# Tokenize
encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)

torch.Size([2000, 128])


In [20]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)
train = dataset["train"]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Get prompt from row 0
text = train[0]["prompt"]

# Tokenize using default settings (NO padding or truncation)
inputs = tokenizer(text, return_tensors="pt")

# Pass through the model
outputs = model(**inputs)

# Print the shape of last_hidden_state
print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [22]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)
train = dataset["train"]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Get prompt from row 0
text = train[0]["prompt"]

# Tokenize (default settings)
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
outputs = model(**inputs)

# Extract the [CLS] embedding (batch 0, token 0)
cls_embedding = outputs.last_hidden_state[0, 0]

# Sum the first 5 values
answer = cls_embedding[:5].sum().item()

print("Sum:", round(answer, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sum: -1.2001


In [23]:
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model with attention outputs enabled
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

# Input sentence
text = "Light-ion fusion is a technique."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
outputs = model(**inputs)

# Show tokens with their indices
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens:")
for i, token in enumerate(tokens):
    print(i, token)

# Find the index of "fusion"
fusion_index = tokens.index("fusion")
print("\nFusion token index:", fusion_index)

# Get attention from the last layer, first head
# Shape: (batch_size, num_heads, seq_len, seq_len)
last_layer_attention = outputs.attentions[-1]

# First batch, first head
head0_attention = last_layer_attention[0, 0]

# Attention from [CLS] (token 0) to "fusion"
attention_weight = head0_attention[0, fusion_index].item()

print("\nAttention Weight:", round(attention_weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens:
0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]

Fusion token index: 4

Attention Weight: 0.1025


In [25]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"  
)
train = dataset["train"]

# Load Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get prompt and Option B from row 0
prompt = train[0]["prompt"]
option_b = train[0]["B"]

# Generate embeddings
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Calculate cosine similarity
similarity = util.cos_sim(prompt_embedding, option_b_embedding)

# Print result
print("Cosine Similarity:", round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine Similarity: 0.7658


In [27]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"  # Change path if needed
)
train = dataset["train"]

labels = ["A", "B", "C", "D", "E"]

# -----------------------------
# MAP@3 Function
# -----------------------------
def mapk(actual, predicted, k=3):
    score = 0.0
    for a, p in zip(actual, predicted):
        if a in p[:k]:
            score += 1.0 / (p.index(a) + 1)
    return score / len(actual)

# ==========================================================
# Pipeline 1 : TF-IDF
# ==========================================================
tfidf_predictions = []

for row in train:
    texts = [row["prompt"], row["A"], row["B"], row["C"], row["D"], row["E"]]

    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(texts)

    sims = cosine_similarity(X[0:1], X[1:]).flatten()

    ranking = np.argsort(sims)[::-1]

    tfidf_predictions.append([labels[i] for i in ranking[:3]])

# ==========================================================
# Pipeline 2 : MiniLM
# ==========================================================
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

minilm_predictions = []

for row in train:
    prompt_embedding = model.encode(row["prompt"], convert_to_tensor=True)

    option_embeddings = model.encode(
        [row["A"], row["B"], row["C"], row["D"], row["E"]],
        convert_to_tensor=True
    )

    similarities = util.cos_sim(prompt_embedding, option_embeddings)[0].cpu().numpy()

    ranking = np.argsort(similarities)[::-1]

    minilm_predictions.append([labels[i] for i in ranking[:3]])

# ==========================================================
# Evaluation
# ==========================================================
actual = [row["answer"] for row in train]

tfidf_map3 = mapk(actual, tfidf_predictions)
minilm_map3 = mapk(actual, minilm_predictions)

print("TF-IDF MAP@3 :", round(tfidf_map3, 6))
print("MiniLM MAP@3 :", round(minilm_map3, 6))

# ==========================================================
# Count improvement
# ==========================================================
count = 0

for gt, tfidf_top3, minilm_top3 in zip(actual, tfidf_predictions, minilm_predictions):

    if (gt not in tfidf_top3) and (gt in minilm_top3):
        count += 1

print("Improved Count:", count)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TF-IDF MAP@3 : 0.326917
MiniLM MAP@3 : 0.423083
Improved Count: 564


In [28]:
from datasets import load_dataset
from transformers import pipeline

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"  # Change path if needed
)
train = dataset["train"]

# Initialize zero-shot classifier (defaults to facebook/bart-large-mnli)
classifier = pipeline("zero-shot-classification")

# Prompt from row 1 (2nd row)
prompt = train[1]["prompt"]

# Candidate labels: Options A, B, C
candidate_labels = [
    train[1]["A"],
    train[1]["B"],
    train[1]["C"]
]

# Perform classification
result = classifier(prompt, candidate_labels)

# Print full result
print(result)

# Print top-ranked label and its probability
print("\nTop Label:", result["labels"][0])
print("Top Probability:", round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [29]:
from datasets import load_dataset
from transformers import pipeline

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)
train = dataset["train"]

# Initialize zero-shot classifier
classifier = pipeline("zero-shot-classification")

# Prompt from row 1 (index 1)
prompt = train[1]["prompt"]

# Candidate labels: A, B, C
candidate_labels = [
    train[1]["A"],
    train[1]["B"],
    train[1]["C"]
]

# Softmax (default)
result_softmax = classifier(prompt, candidate_labels)

# Independent Sigmoids
result_multilabel = classifier(
    prompt,
    candidate_labels,
    multi_label=True
)

# Calculate sums
softmax_sum = sum(result_softmax["scores"])
multilabel_sum = sum(result_multilabel["scores"])

# Absolute difference
difference = abs(softmax_sum - multilabel_sum)

print("Softmax Sum:", softmax_sum)
print("Multi-label Sum:", multilabel_sum)
print("Absolute Difference:", round(difference, 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Softmax Sum: 0.9999999701976776
Multi-label Sum: 0.0005096072964079212
Absolute Difference: 0.9995


In [32]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Text2TextGenerationPipeline

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/smart-mcq-solver-challenge/train.csv"
)
train = dataset["train"]

# Load FLAN-T5 Small
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

generator = Text2TextGenerationPipeline(
    model=model,
    tokenizer=tokenizer
)

# Construct the exact prompt
prompt = (
    f"Question: {train[0]['prompt']}. "
    f"Is the correct answer A: {train[0]['A']} or B: {train[0]['B']}? "
    f"Answer with just the letter A or B."
)

# Generate
result = generator(prompt, max_new_tokens=5)

print("Generated Text:", result[0]["generated_text"])

ImportError: cannot import name 'Text2TextGenerationPipeline' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [33]:
import transformers
print(transformers.__version__)

5.0.0


In [34]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load dataset
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)
train = dataset["train"]

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# Build the prompt
prompt = (
    f"Question: {train[0]['prompt']}. "
    f"Is the correct answer A: {train[0]['A']} or B: {train[0]['B']}? "
    f"Answer with just the letter A or B."
)

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt")

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

# Decode
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Generated Text:", answer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generated Text: B
